# WBR investigation — week 6

**Question:** Why did ad revenue and CTR fall this week?

Playbook: check the data → revenue bridge → mix vs rate → drill down one dimension at a time → find where it concentrates → match to an event → quantify → recommend. All data is simulated.

In [ ]:
import sys, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sys.path.insert(0, '.')
import wbr_lib as w
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid')
WEEK, PRIOR = 6, 6 - 1
print(f'WBR investigation: week {WEEK} vs week {PRIOR}  (ALL DATA SIMULATED)')

## 1. Check the data first
Days complete? Billed clicks vs logged clicks? Orders per impression steady?

In [ ]:
dq = w.data_checks(WEEK)
dq

## 2. Scorecard — the 6 JD metrics

In [ ]:
w.scorecard_table(WEEK)

In [ ]:
w.scorecard(WEEK)[['metric','value','prior_week_value','wow_pct','four_week_avg','vs_4wk_pct']]

## 3. Revenue bridge — which term of requests × fill × CTR × price moved?

In [ ]:
br = w.bridge(WEEK)
br

## 4. Mix or rate? CTR across surface × position × platform

In [ ]:
ps = w.platform_slots((PRIOR, WEEK))
mr = w.mix_rate(ps, ['surface', 'slot_position', 'platform'], w0=PRIOR, w1=WEEK)
print(f"CTR {mr.attrs['total_prior']:.4%} -> {mr.attrs['total_current']:.4%}; mix {mr.attrs['mix']:+.5f}, rate {mr.attrs['rate']:+.5f}")
mr.head(8)

## 5. Drill down — food listings by platform and position

In [ ]:
food = ps[ps.surface == 'food_listing'].assign(ctr=lambda d: d.clicks / d.impressions)
food.pivot_table(index=['platform', 'slot_position'], columns='week_number', values='ctr')

In [ ]:
tr = w.platform_slots(tuple(range(1, 27)), 'food_listing')
tr = tr.assign(seg=lambda d: d.platform + ' p' + d.slot_position.astype(str)).groupby(['week_number','seg'])[['clicks','impressions']].sum()
ax = (tr.clicks / tr.impressions).unstack().plot(figsize=(11, 4), title='Food listings CTR by platform x position')
ax.axvspan(5.5, 7.5, alpha=.1)

## 6. Quantify — lost clicks × CPC

In [ ]:
seg = food[(food.platform == 'android') & (food.slot_position == 1)].set_index('week_number')
base_ctr = seg.loc[PRIOR, 'clicks'] / seg.loc[PRIOR, 'impressions']
lost_clicks = base_ctr * seg.loc[WEEK, 'impressions'] - seg.loc[WEEK, 'clicks']
cpc = w.scorecard(WEEK, 'food_listing').set_index('metric').loc['cpc', 'value']
print(f'Lost clicks on Android slot 1: {lost_clicks:,.0f}; at CPC {w.inr(cpc)} = {w.inr(lost_clicks * cpc)} of revenue this week')